# 🇲🇦 Morocco Job Market — Salary Prediction
> From raw candidate profiles to a production-ready salary estimator, following a 10-step professional ML pipeline blueprint.

| Step | Stage |
|------|-------|
| 1 | Environment & Ingestion |
| 2 | Exploratory Data Analysis |
| 3 | Preprocessing & Standardization |
| 4 | Feature Engineering |
| 5 | Validation Boundary (Train/Test Split) |
| 6 | Modeling (Regression) |
| 7 | Evaluation Scorecard |
| 8 | Hyperparameter Optimization |
| 9 | Model Serialization |
| 10 | Production Deployment (FastAPI) |

**Goal:** predict the **monthly gross salary (MAD)** of a professional in the Moroccan job market from their profile: experience, education, city, sector, company size, contract type, languages, and seniority.

---
## Step 1 — Environment & Ingestion
**Objective:** Load (or generate) the raw data and configure the required analytical libraries.

> ⚠️ **Pitfall:** Training on a dataset that isn't representative of the real population — e.g. only Casablanca tech salaries — produces a model that fails everywhere else.
> ✅ **Best Practice:** Document exactly how the data was sourced so anyone reviewing the project can judge its reliability.

This notebook ships with a **reproducible synthetic data generator** calibrated to publicly observable patterns in the Moroccan job market (higher salaries in Casablanca/Rabat, in IT/Finance, in larger companies, and with seniority). To use **real** data, simply replace the generator with:
```python
df = pd.read_csv('your_morocco_salaries.csv')
```
as long as it keeps the same column schema.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Synthetic Data Generator — Moroccan Job Market ────────────────────────────
N = 1800

cities = ['Casablanca', 'Rabat', 'Marrakech', 'Tanger', 'Fes', 'Agadir', 'Oujda', 'Meknes', 'Kenitra']
city_weights = [0.22, 0.14, 0.12, 0.11, 0.09, 0.09, 0.07, 0.08, 0.08]
city_multiplier = {'Casablanca': 1.18, 'Rabat': 1.12, 'Marrakech': 1.02, 'Tanger': 1.07,
                    'Fes': 0.93, 'Agadir': 0.95, 'Oujda': 0.85, 'Meknes': 0.87, 'Kenitra': 0.90}

sectors = ['IT & Tech', 'Finance & Banking', 'Engineering & Industry', 'Marketing & Communication',
           'Healthcare', 'Tourism & Hospitality', 'Education', 'Public Administration']
sector_weights = [0.18, 0.13, 0.16, 0.12, 0.10, 0.11, 0.10, 0.10]
sector_multiplier = {'IT & Tech': 1.38, 'Finance & Banking': 1.28, 'Engineering & Industry': 1.18,
                      'Marketing & Communication': 1.06, 'Healthcare': 1.12, 'Tourism & Hospitality': 0.90,
                      'Education': 0.85, 'Public Administration': 0.95}

education_levels = ['Bac', 'Bac+2', 'Licence (Bac+3)', 'Master (Bac+5)', 'Doctorat']
education_weights = [0.12, 0.22, 0.30, 0.28, 0.08]
education_base = {'Bac': 4200, 'Bac+2': 5600, 'Licence (Bac+3)': 7000,
                   'Master (Bac+5)': 9600, 'Doctorat': 12500}

company_sizes = ['Startup', 'PME', 'Grande Entreprise', 'Multinationale']
company_weights = [0.18, 0.37, 0.27, 0.18]
company_multiplier = {'Startup': 0.90, 'PME': 1.0, 'Grande Entreprise': 1.15, 'Multinationale': 1.32}

contract_types = ['CDI', 'CDD', 'Freelance', 'Stage']
contract_weights = [0.62, 0.18, 0.12, 0.08]
contract_multiplier = {'CDI': 1.10, 'CDD': 0.95, 'Freelance': 1.0, 'Stage': 0.42}

df = pd.DataFrame({
    'years_experience': np.clip(np.random.exponential(scale=6, size=N), 0, 35).round(1),
    'education_level': np.random.choice(education_levels, size=N, p=education_weights),
    'city': np.random.choice(cities, size=N, p=city_weights),
    'sector': np.random.choice(sectors, size=N, p=sector_weights),
    'company_size': np.random.choice(company_sizes, size=N, p=company_weights),
    'contract_type': np.random.choice(contract_types, size=N, p=contract_weights),
    'num_languages': np.random.choice([1, 2, 3, 4], size=N, p=[0.10, 0.40, 0.38, 0.12]),
    'management_role': np.random.binomial(1, 0.18, size=N),
})

# Inject a small share of missing values (realistic — surveys are never 100% complete)
for col in ['years_experience', 'num_languages']:
    mask = np.random.rand(N) < 0.02
    df.loc[mask, col] = np.nan

def compute_salary(row):
    base = education_base[row['education_level']]
    exp = 0 if pd.isna(row['years_experience']) else row['years_experience']
    nlang = 1 if pd.isna(row['num_languages']) else row['num_languages']
    salary = base
    salary += exp * 320 * sector_multiplier[row['sector']]
    salary *= city_multiplier[row['city']]
    salary *= sector_multiplier[row['sector']]
    salary *= company_multiplier[row['company_size']]
    salary *= contract_multiplier[row['contract_type']]
    salary += (nlang - 1) * 280
    salary += row['management_role'] * 2600
    noise = np.random.normal(0, salary * 0.07)
    return max(2500, salary + noise)

df['monthly_salary_mad'] = df.apply(compute_salary, axis=1).round(0)

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

---
## Step 2 — Exploratory Data Analysis (The Diagnostics)
**Objective:** Uncover the underlying structure, distributions, and hidden relationships within the raw data before any modification occurs.

> ⚠️ **Pitfall:** Skipping target variable analysis — leads to undetected skew that harms model performance.
> ✅ **Best Practice:** Always look at the target's distribution *and* how it breaks down across the main categorical drivers (here: city and sector).

In [ ]:
# ── 2.1 Structural Profile ────────────────────────────────────────────────────
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")
print(f"  Missing : {df.isnull().sum().sum()}")
print()
print(df.describe(include='all').T)

In [ ]:
# ── 2.2 Target Distribution ───────────────────────────────────────────────────
plt.figure(figsize=(8, 4.5))
sns.histplot(df['monthly_salary_mad'], bins=40, color='steelblue', edgecolor='white')
plt.title('Step 2 — Monthly Salary Distribution (MAD)', fontsize=13, fontweight='bold')
plt.xlabel('Monthly salary (MAD)')
plt.tight_layout()
plt.show()

print(f"Target skew: {df['monthly_salary_mad'].skew():.3f}")

In [ ]:
# ── 2.3 Salary by City & Sector ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

order_city = df.groupby('city')['monthly_salary_mad'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='city', y='monthly_salary_mad', order=order_city, ax=axes[0], palette='Blues_r')
axes[0].set_title('Salary by City')
axes[0].tick_params(axis='x', rotation=45)

order_sector = df.groupby('sector')['monthly_salary_mad'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='sector', y='monthly_salary_mad', order=order_sector, ax=axes[1], palette='Greens_r')
axes[1].set_title('Salary by Sector')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Step 2 — Salary Breakdown by Category', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Numeric Correlations ──────────────────────────────────────────────────
numeric_cols = ['years_experience', 'num_languages', 'management_role', 'monthly_salary_mad']
corr = df[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.4)
plt.title('Step 2 — Numeric Correlations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3 — Preprocessing & Standardization
**Objective:** Cleanse anomalies and transform human-readable data (mixed numeric + categorical) into a uniform mathematical format.

> ⚠️ **Pitfall:** Applying `fit_transform()` to the entire dataset before splitting — causes **critical data leakage**.
> ✅ **Best Practice:** Build the full numeric + categorical logic inside a single `ColumnTransformer`, but only **fit** it after the train/test split (Step 5).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ── Define Features & Target ──────────────────────────────────────────────────
TARGET = 'monthly_salary_mad'
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print(f"Target mean : {y.mean():,.0f} MAD")
print(f"Target std  : {y.std():,.0f} MAD")

---
## Step 4 — Feature Engineering
**Objective:** Construct new, highly predictive variables from existing data to give the model stronger learning signals.

> ⚠️ **Pitfall:** Creating features that rely on future information (target leakage).
> ✅ **Best Practice:** Consult domain knowledge — seniority bands and "major economic hub" flags are well known salary drivers in the Moroccan labor market.

In [ ]:
# ── Domain-driven feature engineering ────────────────────────────────────────
def engineer(X_in):
    X_e = X_in.copy()

    # 1. Seniority band (non-linear experience effect)
    bins = [-0.1, 3, 7, 15, 100]
    labels = ['Junior (0-3y)', 'Mid (3-7y)', 'Senior (7-15y)', 'Expert (15y+)']
    X_e['experience_level'] = pd.cut(X_e['years_experience'].fillna(0), bins=bins, labels=labels)

    # 2. Major economic hub flag (Casablanca / Rabat / Marrakech / Tanger)
    X_e['is_major_city'] = X_e['city'].isin(['Casablanca', 'Rabat', 'Marrakech', 'Tanger']).astype(int)

    # 3. Languages × management interaction (multilingual managers are paid a premium)
    X_e['languages_x_management'] = X_e['num_languages'].fillna(1) * X_e['management_role']

    return X_e

X_eng = engineer(X)
X_eng.head()

---
## Step 5 — The Validation Boundary
**Objective:** Isolate a portion of the data to evaluate the model's performance on completely unseen profiles.

> ✅ **Best Practice:** Always set `random_state` so colleagues can replicate your exact data split, and only **fit** the preprocessor after the split.

In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = ['years_experience', 'num_languages', 'management_role',
                     'languages_x_management', 'is_major_city']
categorical_features = ['education_level', 'city', 'sector', 'company_size',
                         'contract_type', 'experience_level']

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])

# ── Split FIRST, then fit the preprocessor ───────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_eng, y,
    test_size=0.20,
    random_state=42      # guarantees reproducibility
)

# ── NOW fit the preprocessor only on training data ────────────────────────────
X_train_t = preprocessor.fit_transform(X_train)
X_test_t  = preprocessor.transform(X_test)   # transform only — no fit!

print(f"Train shape: {X_train_t.shape}  |  Test shape: {X_test_t.shape}")

---
## Step 6 — The Modeling Matrix
**Objective:** Select and train the right regression algorithm.

This is a **regression** problem — the target (monthly salary in MAD) is a continuous number.
We train three candidate models and compare them before selecting the winner.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

candidates = {
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=150, random_state=42)
}

results = {}
for name, model in candidates.items():
    model.fit(X_train_t, y_train)
    preds = model.predict(X_test_t)
    results[name] = {
        'model': model,
        'rmse': np.sqrt(mean_squared_error(y_test, preds)),
        'mae': mean_absolute_error(y_test, preds),
        'r2': r2_score(y_test, preds)
    }

comparison = pd.DataFrame({k: {m: round(v, 3) for m, v in d.items() if m != 'model'} for k, d in results.items()}).T
print(comparison)

best_name = min(results, key=lambda k: results[k]['rmse'])
best_model = results[best_name]['model']
print(f"\n🏆 Best model: {best_name}")

---
## Step 7 — The Evaluation Scorecard
**Objective:** Measure real-world model performance with the metrics that matter for regression.

| Metric | Formula | Plain English |
|--------|---------|---------------|
| RMSE | √(mean((ŷ − y)²)) | Average prediction error, in MAD |
| MAE | mean(\|ŷ − y\|) | Median-robust error, in MAD |
| R² | 1 − SS_res/SS_tot | % of salary variance explained by the model |

In [ ]:
# ── Detailed evaluation of the best model ────────────────────────────────────
preds = best_model.predict(X_test_t)
residuals = y_test - preds

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae  = mean_absolute_error(y_test, preds)
r2   = r2_score(y_test, preds)

print("=" * 45)
print(f"  EVALUATION SCORECARD — {best_name}")
print("=" * 45)
print(f"  RMSE : {rmse:,.0f} MAD  (lower = better)")
print(f"  MAE  : {mae:,.0f} MAD  (lower = better)")
print(f"  R²   : {r2:.4f}  (closer to 1 = better)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(y_test, preds, alpha=0.3, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual salary (MAD)')
axes[0].set_ylabel('Predicted salary (MAD)')
axes[0].set_title('Actual vs Predicted')

axes[1].hist(residuals, bins=40, color='salmon', edgecolor='white')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual (MAD)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance (tree-based models only) ───────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    feature_names = preprocessor.get_feature_names_out()
    importances = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=True).tail(15)

    plt.figure(figsize=(9, 6))
    importances.plot(kind='barh', color='steelblue', edgecolor='white')
    plt.title('Step 7 — Top 15 Feature Importances', fontsize=13, fontweight='bold')
    plt.xlabel('Importance score')
    plt.tight_layout()
    plt.show()

---
## Step 8 — Hyperparameter Optimization
**Objective:** Systematically test algorithmic settings to locate the configuration yielding the lowest error.

> ⚠️ **Pitfall:** Searching too many parameters at once — exponentially long computation.
> ✅ **Best Practice:** Use `RandomizedSearchCV` to efficiently sample a wide configuration space instead of exhaustively testing every combination.

We continue the optimization with **Random Forest**: it's tunable, interpretable (feature importances), and competitive on this data — a solid choice to take to production regardless of which model narrowly won the raw comparison above.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators'      : [100, 150, 200],
    'max_depth'         : [8, 12, 16, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=1),   # n_jobs=1 here: the search itself parallelizes
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train_t, y_train)

optimized_model = random_search.best_estimator_
preds_opt = optimized_model.predict(X_test_t)
rmse_opt = np.sqrt(mean_squared_error(y_test, preds_opt))
r2_opt   = r2_score(y_test, preds_opt)

print("Best parameters:", random_search.best_params_)
print(f"Optimized RMSE : {rmse_opt:,.0f} MAD")
print(f"Optimized R²   : {r2_opt:.4f}")

---
## Step 9 — Model Serialization
**Objective:** Export the fully trained model **and** its preprocessing pipeline as a static, transportable artifact.

> ⚠️ **Pitfall:** Saving only the model and forgetting the fitted encoders/scalers — incoming raw data can't be transformed.
> ✅ **Best Practice:** Version-control saved models alongside the exact Git commit used to generate them.

In [ ]:
import joblib
from datetime import date

# ── Bundle model + preprocessor together ─────────────────────────────────────
artifact = {
    'model'                 : optimized_model,
    'preprocessor'          : preprocessor,             # fitted ColumnTransformer
    'numeric_features'      : numeric_features,
    'categorical_features'  : categorical_features,
    'target'                : TARGET,
    'trained_on'            : str(date.today()),
    'r2_test'               : round(r2_opt, 4),
    'rmse_test'             : round(rmse_opt, 2),
}

filename = f"morocco_salary_model_v1.0_{date.today().strftime('%Y%m%d')}.pkl"
joblib.dump(artifact, filename)
print(f"Artifact saved as: {filename}")
print(f"  R² on test set   : {artifact['r2_test']}")
print(f"  RMSE on test set : {artifact['rmse_test']:,.0f} MAD")

# ── Verify round-trip load ────────────────────────────────────────────────────
loaded = joblib.load(filename)
sample_raw = engineer(X_test.iloc[:3])
sample_pred = loaded['model'].predict(loaded['preprocessor'].transform(sample_raw))
print(f"\nRound-trip load verified. Sample predictions (MAD): {sample_pred.round(0)}")

---
## Step 10 — Production Deployment (FastAPI Stub)
**Objective:** Integrate the serialized model into a live software architecture to generate real-time predictions.

> ⚠️ **Pitfall:** Assuming the model will perform flawlessly forever. Salary scales shift over time with inflation and labor market changes (**model drift**).
> ✅ **Best Practice:** Log every prediction in production to actively monitor for performance degradation, and retrain periodically on fresh data.

In [ ]:
# ── Production API Stub (FastAPI) ─────────────────────────────────────────────
# Save this as app.py and run with: uvicorn app:app --reload

fastapi_code = '''
# app.py  —  Morocco Salary Prediction API
# Run: uvicorn app:app --reload
# Install: pip install fastapi uvicorn joblib pandas numpy

from fastapi import FastAPI
from pydantic import BaseModel
import joblib, pandas as pd, logging, datetime

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="Morocco Salary Prediction API", version="1.0")

# Load the most recently generated artifact at startup
import glob as _glob
_pkls = sorted(_glob.glob("morocco_salary_model_v1.0_*.pkl"))
if not _pkls:
    raise FileNotFoundError("No model artifact found. Run Step 9 of the notebook first.")
artifact = joblib.load(_pkls[-1])
model = artifact["model"]
preprocessor = artifact["preprocessor"]


class CandidateProfile(BaseModel):
    years_experience: float
    education_level: str        # Bac | Bac+2 | Licence (Bac+3) | Master (Bac+5) | Doctorat
    city: str                   # Casablanca | Rabat | Marrakech | Tanger | Fes | Agadir | Oujda | Meknes | Kenitra
    sector: str                 # IT & Tech | Finance & Banking | Engineering & Industry | ...
    company_size: str           # Startup | PME | Grande Entreprise | Multinationale
    contract_type: str          # CDI | CDD | Freelance | Stage
    num_languages: int
    management_role: int        # 0 or 1


def engineer(df):
    bins = [-0.1, 3, 7, 15, 100]
    labels = ["Junior (0-3y)", "Mid (3-7y)", "Senior (7-15y)", "Expert (15y+)"]
    df["experience_level"] = pd.cut(df["years_experience"].fillna(0), bins=bins, labels=labels)
    df["is_major_city"] = df["city"].isin(["Casablanca", "Rabat", "Marrakech", "Tanger"]).astype(int)
    df["languages_x_management"] = df["num_languages"].fillna(1) * df["management_role"]
    return df


@app.post("/predict")
def predict(profile: CandidateProfile):
    raw = pd.DataFrame([profile.dict()])
    engineered = engineer(raw)
    X_t = preprocessor.transform(engineered)
    prediction = float(model.predict(X_t)[0])

    logger.info(f"{datetime.datetime.now().isoformat()} | prediction={prediction:.0f} MAD | input={profile.dict()}")

    return {"predicted_monthly_salary_mad": round(prediction, 0), "currency": "MAD"}


@app.get("/health")
def health():
    return {"status": "ok", "model_r2": artifact.get("r2_test")}
'''

with open("app.py", "w") as f:
    f.write(fastapi_code)

print("app.py generated. Run with: uvicorn app:app --reload")

---
## The Continuous Lifecycle

> A professional pipeline is not a one-and-done script. It is an automated engine designed to be run repeatedly as new salary data flows back from the real market.

```
Ingestion → EDA → Preprocessing → Engineering → Split
    ↑                                               ↓
New Survey Data                                 Modeling
    ↑                                               ↓
Drift Detected ← Monitoring ← Deployment ← Serialization
```

### Professional's Diagnostic Checklist
- Re-survey the job market periodically (annual inflation alone shifts salary bands).
- Track prediction logs for drift — a sudden gap between predicted and reported salaries signals the model needs retraining.
- Keep the preprocessor and model bundled together; never deploy one without the other.